In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

In [ ]:
tabelas_delta = ['albuns', 'artistas', 'musicas', 'reproducoes', 'usuarios']

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

In [ ]:
print('=== ARTISTAS ===')
spark.sql('SELECT * FROM artistas ORDER BY id').show()

print('=== USUARIOS ===')
spark.sql('SELECT * FROM usuarios ORDER BY id').show()

print('=== MUSICAS ===')
spark.sql('SELECT * FROM musicas ORDER BY id').show()

In [ ]:
# Contagem de registros por tabela
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<15} {count:>10}')

## INSERT — novo artista e novo usuário

In [ ]:
spark.sql("""
    INSERT INTO artistas VALUES (6, 'Novo Artista', 'Brasil', 'MPB', '2024-01-01')
""")

spark.sql("""
    INSERT INTO usuarios VALUES (6, 'Felipe Teste', 'felipe@teste.com', 'premium', '2024-05-01')
""")

print('Após INSERT:')
spark.sql('SELECT * FROM artistas ORDER BY id').show()
spark.sql('SELECT * FROM usuarios ORDER BY id').show()

## UPDATE — alterar plano de usuário

In [ ]:
DeltaTable.forName(spark, 'usuarios').update(
    condition="plano = 'gratuito'",
    set={"plano": "'basico'"}
)

print('Após UPDATE (gratuito → basico):')
spark.sql('SELECT id, nome, plano FROM usuarios ORDER BY id').show()

## DELETE — remover artista inserido

In [ ]:
DeltaTable.forName(spark, 'artistas').delete("id = 6")

print('Após DELETE (artista id=6):')
spark.sql('SELECT * FROM artistas ORDER BY id').show()

## MERGE (UPSERT) — atualizar ou inserir músicas

In [ ]:
from pyspark.sql import Row

# Fonte: id=1 existe (atualiza duração), id=99 não existe (insere)
atualizacoes = spark.createDataFrame([
    Row(id=1, album_id=1, titulo='Música Atualizada', duracao_segundos=999, numero_faixa=1),
    Row(id=99, album_id=2, titulo='Música Nova', duracao_segundos=180, numero_faixa=10),
])

(
    DeltaTable.forName(spark, 'musicas')
    .alias('alvo')
    .merge(atualizacoes.alias('src'), 'alvo.id = src.id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print('Após MERGE:')
spark.sql('SELECT * FROM musicas ORDER BY id').show()

## Time Travel — histórico de versões

In [ ]:
print('=== HISTÓRICO (musicas) ===')
DeltaTable.forName(spark, 'musicas').history().select(
    'version', 'timestamp', 'operation', 'operationParameters'
).show(truncate=False)

print('=== VERSÃO 0 (estado original) ===')
spark.read.format('delta').option('versionAsOf', 0).table('musicas').show()